In [1]:
import sys
sys.path.append('../') 
from visualisation import *
import xarray as xr
import concurrent.futures

In [4]:
os.path.getsize('G-NAF/')/1024

4.0

In [3]:
df_address = pd.read_csv('bom_postcodes_points.csv')
df_address['longitude'] = df_address['longitude'].round(2).astype(str)
df_address['latitude'] = df_address['latitude'].round(2).astype(str)

In [8]:
5068 in df_address['nearest_postcode'].values

True

In [3]:
bom_path = "/home/hossein/CICCADA/BOM_NCI/2022/**/**/*.nc"
files = glob(bom_path)
len(files)

3090

In [4]:
def process_file(file):
    try:
        df = xr.open_dataset(file).to_dataframe().reset_index(drop=False)
        df = df.dropna(subset='surface_global_irradiance').reset_index(drop=True)
        df['longitude'] = df['longitude'].round(2).astype(str)
        df['latitude'] = df['latitude'].round(2).astype(str)
        df = df.merge(df_address, on=['latitude', 'longitude'], how='inner')
        df.drop(columns=['latitude', 'longitude', 'crs', 'julian_date'], inplace=True)
        return df
    except Exception as e:
        print(f"Error processing file {file}: {e}")
        return None

num_cores = 12 


In [5]:

with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
    df_list = list(executor.map(process_file, files))

df = pd.concat(df_list, axis=0).reset_index(drop=True)


In [8]:
dfbk = df.copy()

In [52]:
df = dfbk.copy()

In [51]:
df.head()

,time,postcode,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,distance_km
0,2022-12-31 18:30:00,872.0,NaN,NaN,NaN,1.0,6.614286,4.747143,-20.595714,130.905714,1.365766
1,2022-12-31 18:30:00,5000.0,NaN,NaN,NaN,1.0,0.000000,0.000000,-10.500000,128.000000,0.182848
2,2022-12-31 18:30:00,5007.0,NaN,NaN,NaN,1.0,0.000000,0.000000,-10.600000,128.000000,0.089172
3,2022-12-31 18:30:00,5008.0,NaN,NaN,NaN,1.0,0.000000,0.000000,-10.600000,128.000000,0.096957
4,2022-12-31 18:30:00,5010.0,NaN,NaN,NaN,1.0,0.000000,0.000000,-10.600000,128.000000,0.107950


In [60]:
df.query(f"time=='2022-12-31 18:30:00' and nearest_postcode==872 and direct_normal_irradiance==direct_normal_irradiance")

,time,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,nearest_postcode,distance_km


In [6]:
df.rename(columns={'nearest_postcode': 'postcode'}, inplace=True)
df = df.groupby(['time', 'postcode']).mean().reset_index()

In [12]:
df['postcode'].unique().shape

(331,)

In [82]:
df0 = df.query(f"surface_global_irradiance==surface_global_irradiance").groupby('postcode').agg({'time': ['min', 'max', 'count']}).reset_index()
df0.columns = ['_'.join(col).strip() for col in df0.columns.values]
df0

,postcode_,time_min,time_max,time_count
0,872.0,2022-12-31 20:10:00,2023-01-31 10:00:00,2561
1,5000.0,2022-12-31 19:30:00,2023-01-31 09:40:00,2607
2,5007.0,2022-12-31 19:30:00,2023-01-31 09:40:00,2606
3,5008.0,2022-12-31 19:30:00,2023-01-31 09:40:00,2606
4,5010.0,2022-12-31 19:30:00,2023-01-31 09:40:00,2606
...,...,...,...,...
326,5731.0,2022-12-31 19:40:00,2023-01-31 09:40:00,2565
327,5732.0,2022-12-31 19:40:00,2023-01-31 09:30:00,2563
328,5733.0,2022-12-31 19:50:00,2023-01-31 09:40:00,2539
329,5734.0,2022-12-31 19:50:00,2023-01-31 09:40:00,2550


In [10]:
df0 = df.sort_values(by='time')
df0.groupby('postcode').apply(lambda x:x.diff().quantile(.99))

0.99,time,postcode,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,distance_km
postcode,,,,,,,,,,,
872.0,0 days 12:02:17.999999999,0.0,47.324413,161.403437,41.541106,0.0000,1.303286,9.506576,2.231429,160.942929,0.042004
5000.0,0 days 12:41:29.999999999,0.0,201.444750,671.167500,202.073250,0.0000,6.000000,42.492000,2.100000,352.915000,0.000000
5007.0,0 days 12:41:18,0.0,202.392700,656.927300,209.264600,0.0000,6.000000,35.232700,2.050000,352.856500,0.000000
5008.0,0 days 12:41:18,0.0,193.073400,637.714500,210.147750,0.0000,6.000000,35.484450,2.050000,352.856500,0.000000
5010.0,0 days 12:41:18,0.0,211.063000,743.508600,261.616000,0.0000,6.000000,28.991700,2.100000,352.900000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
5731.0,0 days 12:20:00,0.0,43.136646,153.867906,35.886097,0.0125,1.190918,7.150213,2.176531,203.119102,0.032697
5732.0,0 days 12:30:00,0.0,44.603047,195.051512,42.079313,0.0000,1.235647,7.584326,2.149412,350.314082,0.007831
5733.0,0 days 12:20:00,0.0,43.834639,164.397310,38.883386,0.0000,1.189189,8.490522,2.191892,351.575000,0.045350


In [7]:
df.to_csv('/home/hossein/CICCADA/BOM_NCI/2022/NCI_processed_grouped_Nov.csv', index=False)

In [8]:
os.path.getsize('/home/hossein/CICCADA/BOM_NCI/2022/NCI_processed_grouped_Nov.csv') / (1024 * 1024)

113.2850170135498

In [6]:
df = pd.read_csv('/home/hossein/CICCADA/BOM_NCI/2022/NCI_processed_grouped_Nov.csv')

In [7]:
5068 in df['postcode'].unique()

True

In [22]:
df['time'].min(), df['time'].max(), df['postcode'].unique().shape

('2022-12-31 19:10:00', '2023-01-01 10:20:00', (331,))

In [18]:
df = pd.concat([pd.read_csv(f) for f in glob('/home/hossein/CICCADA/BOM_NCI/2023/NCI_processed_grouped_*.csv')], axis=0).reset_index(drop=True)

In [21]:
df['time'].min(), df['time'].max(), df['postcode'].unique().shape

('2022-12-31 19:10:00', '2023-12-31 10:20:00', (331,))

In [22]:
df.to_csv('/home/hossein/CICCADA/BOM_NCI/2023/NCI_processed_grouped_all.csv', index=False)